# Candidate Dataset EDA

Initial exploratory data analysis for `data/raw/candidates.json`.

This notebook checks dataset shape, missing values, schema coverage, common skills, and experience distributions.

In [ ]:
from collections import Counter, defaultdict
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

## Load Raw Dataset

In [ ]:
DATA_PATH = Path("../data/raw/candidates.json")

with DATA_PATH.open("r", encoding="utf-8") as f:
    candidates = json.load(f)

type(candidates), len(candidates)

In [ ]:
# Candidate-level normalized table for quick tabular inspection.
df = pd.json_normalize(candidates, sep=".")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()

## Shape and Top-Level Structure

In [ ]:
top_level_keys = sorted({key for row in candidates for key in row.keys()})

top_level_summary = pd.DataFrame(
    {
        "field": top_level_keys,
        "present_count": [sum(key in row for row in candidates) for key in top_level_keys],
        "example_type": [type(next(row[key] for row in candidates if key in row)).__name__ for key in top_level_keys],
    }
)
top_level_summary["present_pct"] = top_level_summary["present_count"] / len(candidates)
top_level_summary

In [ ]:
list_lengths = []

for row in candidates:
    candidate_id = row.get("candidate_id")
    for key, value in row.items():
        if isinstance(value, list):
            list_lengths.append(
                {
                    "candidate_id": candidate_id,
                    "field": key,
                    "length": len(value),
                }
            )

list_lengths_df = pd.DataFrame(list_lengths)
list_lengths_df.groupby("field")["length"].describe()

## Nulls and Missing Fields

In [ ]:
null_summary = (
    df.isna()
    .sum()
    .rename("null_count")
    .to_frame()
    .assign(null_pct=lambda x: x["null_count"] / len(df))
    .query("null_count > 0")
    .sort_values(["null_count", "null_pct"], ascending=False)
)

null_summary

In [ ]:
def iter_paths(obj, prefix=""):
    """Yield leaf and container paths from a nested JSON object."""
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key
            yield path, value
            yield from iter_paths(value, path)
    elif isinstance(obj, list):
        list_path = f"{prefix}[]"
        yield list_path, obj
        for item in obj:
            yield from iter_paths(item, list_path)


path_counts = Counter()
type_counts = defaultdict(Counter)
null_counts = Counter()

for row in candidates:
    seen_paths = set()
    for path, value in iter_paths(row):
        seen_paths.add(path)
        type_counts[path][type(value).__name__] += 1
        if value is None:
            null_counts[path] += 1
    path_counts.update(seen_paths)

schema_df = pd.DataFrame(
    [
        {
            "path": path,
            "present_count": count,
            "present_pct": count / len(candidates),
            "null_count": null_counts[path],
            "types": dict(type_counts[path]),
        }
        for path, count in sorted(path_counts.items())
    ]
)

schema_df.sort_values(["present_pct", "path"], ascending=[True, True]).head(30)

In [ ]:
required_top_level_fields = [
    "candidate_id",
    "profile",
    "career_history",
    "education",
    "skills",
    "certifications",
    "languages",
    "redrob_signals",
]

missing_required = []
for row in candidates:
    for field in required_top_level_fields:
        if field not in row or row[field] is None:
            missing_required.append({"candidate_id": row.get("candidate_id"), "missing_field": field})

pd.DataFrame(missing_required)

## Schema Map

In [ ]:
schema_df.sort_values("path").reset_index(drop=True)

In [ ]:
nested_object_keys = []

for row in candidates:
    for top_key, value in row.items():
        if isinstance(value, dict):
            nested_object_keys.append(
                {
                    "section": top_key,
                    "keys": tuple(sorted(value.keys())),
                }
            )
        elif isinstance(value, list):
            item_keys = tuple(
                sorted({key for item in value if isinstance(item, dict) for key in item.keys()})
            )
            nested_object_keys.append(
                {
                    "section": f"{top_key}[]",
                    "keys": item_keys,
                }
            )

section_schema = (
    pd.DataFrame(nested_object_keys)
    .drop_duplicates()
    .sort_values("section")
    .reset_index(drop=True)
)
section_schema

## Skills Analysis

In [ ]:
skills_records = []

for row in candidates:
    candidate_id = row.get("candidate_id")
    years_of_experience = row.get("profile", {}).get("years_of_experience")
    for skill in row.get("skills", []):
        skills_records.append(
            {
                "candidate_id": candidate_id,
                "candidate_years_of_experience": years_of_experience,
                **skill,
            }
        )

skills_df = pd.DataFrame(skills_records)
print(f"Skill rows: {len(skills_df):,}")
skills_df.head()

In [ ]:
top_skills = (
    skills_df.groupby("name")
    .agg(
        candidate_count=("candidate_id", "nunique"),
        avg_proficiency=("proficiency", lambda s: s.mode().iat[0] if not s.mode().empty else None),
        median_duration_months=("duration_months", "median"),
        total_endorsements=("endorsements", "sum"),
    )
    .sort_values(["candidate_count", "total_endorsements"], ascending=False)
)

top_skills.head(25)

In [ ]:
skills_df["proficiency"].value_counts(dropna=False).rename_axis("proficiency").reset_index(name="skill_count")

## Experience Distributions

In [ ]:
experience = df[
    [
        "candidate_id",
        "profile.years_of_experience",
        "profile.current_title",
        "profile.current_industry",
        "profile.country",
    ]
].rename(
    columns={
        "profile.years_of_experience": "years_of_experience",
        "profile.current_title": "current_title",
        "profile.current_industry": "current_industry",
        "profile.country": "country",
    }
)

experience["experience_band"] = pd.cut(
    experience["years_of_experience"],
    bins=[0, 2, 5, 8, 12, 20, float("inf")],
    labels=["0-2", "2-5", "5-8", "8-12", "12-20", "20+"],
    right=False,
)

experience.describe(include="all")

In [ ]:
experience["experience_band"].value_counts().sort_index().rename_axis("experience_band").reset_index(name="candidate_count")

In [ ]:
experience.groupby("current_title")["years_of_experience"].agg(["count", "mean", "median", "min", "max"]).sort_values(
    ["count", "median"], ascending=False
)

## Career History Durations

In [ ]:
career_records = []

for row in candidates:
    for job in row.get("career_history", []):
        career_records.append({"candidate_id": row.get("candidate_id"), **job})

career_df = pd.DataFrame(career_records)
career_df.head()

In [ ]:
career_df["duration_years"] = career_df["duration_months"] / 12

career_df["duration_years"].describe().to_frame().T

In [ ]:
career_df.groupby("is_current")["duration_months"].agg(["count", "mean", "median", "min", "max"])